In [ ]:
# configs.py

# S3 Bucket and Folder configuration
BUCKET = "cubix-chicago-taxi-bb-v2"

# Raw data folders
RAW_TAXI_FOLDER = "raw_data/to_processed/taxi"
RAW_WEATHER_FOLDER = "raw_data/to_processed/weather"

# Dim data 
DIM_PAYMENT_TYPE_FILE_PATH = "transformed_data/dim_payment_type/dim_payment_type.csv"
DIM_COMPANY_FILE_PATH = "transformed_data/dim_company/dim_company.csv"

In [ ]:
# functions.py

from io import StringIO
import json
from typing import Dict, List

import pandas as pd

def read_file_from_s3(
    s3,
    bucket: str,
    key: str,
    file_format: str = "csv"
):
    """
    Reads a csv or json file from the given S3 bucket.

    :param s3:          S3 client.
    :param bucket:      Name of the S3 Bucket where the file is stored.
    :param key:         Path within the S3 bucket.
    :param file_format: "json" or "csv".
    """
    respone = s3.get_object(Bucket=bucket, Key=key)
    content = respone["Body"].read().decode("utf-8")

    if file_format == "csv":
        return pd.read_csv(StringIO(content))
    elif file_format == "json":
        return json.loads(content)
    else:
        raise ValueError("Unsupported file format, use 'csv' or 'json'.")


def transform_taxi(raw_taxi_data: List[Dict]) -> pd.DataFrame:
    """Perform tranformations on the taxi data.

    1. Drop selected columns.
    2. Drop NULL values accross all columns.
    3. Rename selected columns.
    4. Create "datetime_for_weather" helper column (for dim_weather join).

    :param raw_taxi_data:   The json file holding the daily taxi trips.
    :return:                Transformed taxi trips DataFrame.
    """
    taxi_trips = pd.DataFrame(raw_taxi_data)

    taxi_trips.drop(["pickup_census_tract", "dropoff_census_tract", 
                 "pickup_centroid_location", "dropoff_centroid_location"], axis=1, inplace=True)
    taxi_trips.dropna(inplace=True)

    taxi_trips.rename(columns={"pickup_community_area": "pickup_community_area_id",
                            "dropoff_community_area": "dropoff_community_area_id"}, inplace=True)

    taxi_trips["trip_start_timestamp"] = pd.to_datetime(taxi_trips["trip_start_timestamp"])
    taxi_trips["datetime_for_weather"] = taxi_trips["trip_start_timestamp"].dt.floor("h")

    return taxi_trips


def transform_weather(data: Dict) -> pd.DataFrame:
    """Select and transform weather data.

    :param weather_data:    The daily weather data from the Open Meteo API.
    :return:                Transformed weather pandas DataFrame.
    """

    weather_data = {
        "datetime": data["hourly"]["time"],
        "temperature": data["hourly"]["temperature_2m"],
        "wind_speed": data["hourly"]["wind_speed_10m"],
        "rain": data["hourly"]["rain"],
        "precipitation": data["hourly"]["precipitation"]
    }

    weather_df = pd.DataFrame(weather_data)
    weather_df["datetime"] = pd.to_datetime(weather_df["datetime"])

    return weather_df


def update_dim_table(
    taxi_trips: pd.DataFrame,
    dim_df: pd.DataFrame,
    value_col: str
) -> pd.DataFrame:
    """Extend the dimension DataFrame with new values if there are any.

    :param taxi_trips:  DataFrame with the daily taxi trips.
    :param dim_df:      DataFrame with the dimension data (company, payment_type).
    :param value_col:   Name of the column in dimension DataFrame containing the values.
    :return:            The updated dimension data, if new values are in the taxi data, they will be loaded to it.
    """
    id_col = f"{value_col}_id"

    todays_dim_data = pd.DataFrame(taxi_trips[value_col].unique(), columns=[value_col])
    new_dim_data = todays_dim_data[~todays_dim_data[value_col].isin(dim_df[value_col])]

    if not new_dim_data.empty:
        max_id = dim_df[id_col].max()
        new_dim_data[id_col] = range(max_id + 1, max_id + 1 + len(new_dim_data))
        dim_df = pd.concat([dim_df, new_dim_data], ignore_index=True)

    return dim_df


def update_fact_taxi_trips_with_dim_data(
    taxi_trips: pd.DataFrame,
    dim_payment_type: pd.DataFrame,
    dim_company: pd.DataFrame
) -> pd.DataFrame:
    """Update the fact_taxi_trips DataFrame with the dim_company and dim_payment_type ids, and delete the string columns.

    :param taxi_trips:          The DataFrame with the daily taxi trips.
    :param dim_payment_type:    The payment type dimension table.
    :param dim_company:         The company dimension table.
    :return:                    The taxi trips data, with only payment_type_id and company_id, without company or
                                payment_type values.
    """

    fact_taxi_trips = taxi_trips.merge(dim_payment_type, on="payment_type")
    fact_taxi_trips = fact_taxi_trips.merge(dim_company, on="company")
    fact_taxi_trips.drop(["payment_type", "company"], axis=1, inplace=True)

    return fact_taxi_trips


def _upload_dataframe_to_s3(
    s3,
    bucket: str,
    dataframe: pd.DataFrame,
    path: str
):
    """Uploads a dataframe to the specified S3 path."""
    buffer = StringIO()
    dataframe.to_csv(buffer, index=False)
    df_content = buffer.getvalue()
    s3.put_object(Bucket=bucket, Key=path, Body=df_content)
    print("Uploaded dataframe to S3.")


def upload_dim_to_s3(
    s3,
    bucket: str,
    dim_type: str,
    dataframe: pd.DataFrame
):
    """
    Uploads a dimension table (company or payment_type) to S3.
    Copies the previous version before overwriting the current one.

    :param s3:          S3 client.
    :param bucket:      Name of the S3 bucket.
    :param dim_type:    Name of the dimension (e.g., "company" or "payment_type").
    :param dataframe:   DataFrame to upload.
    :raises ValueError: Raises ValueError when dim_type is not "company" or "payment_type"
    """
    if not dim_type in ["company", "payment_type"]:
        raise ValueError("dim_type must be either 'company' or 'payment_type'")

    current_file_path = f"transformed_data/dim_{dim_type}/dim_{dim_type}.csv"
    previous_version_file_path = f"transformed_data/dimension_table_previous_versions/dim_{dim_type}.csv"

    s3.copy_object(
        Bucket=bucket,
        CopySource={"Bucket": bucket, "Key": current_file_path},
        Key=previous_version_file_path
    )
    print(f"Copied existing version of {dim_type} to previous version folder")

    _upload_dataframe_to_s3(s3, bucket, dataframe, path=current_file_path)


def _move_file_on_s3(
    s3,
    bucket: str,
    source_key: str,
    target_key: str
):
    """Moves a file within S3 by copying it to a new location and deleting the original."""
    s3.copy_object(
        Bucket=bucket,
        CopySource={"Bucket": bucket, "Key": source_key},
        Key=target_key
    )
    s3.delete_object(Bucket=bucket, Key=source_key)
    print(f"Archived raw data.")
    

def upload_and_archive_on_s3(
    s3,
    dataframe: pd.DataFrame,
    bucket: str,
    file_type: str
):
    """
    Uploads a transformed dataset to S3 and archives the corresponding raw file.

    Workflow:
      1. Uploads the transformed DataFrame to it's "transformed" folder with a date-based filename.
      2. Moves the original raw file from "to_processed" to "processed".

    :param s3:          S3 client.
    :param dataframe:   Transformed DataFrame to upload.
    :param bucket:      S3 bucket name.
    :param file_type:   Type of data: "taxi" or "weather".
    """
    match file_type:
        case "taxi":
            formatted_date = dataframe["datetime_for_weather"].dt.strftime("%Y-%m-%d").iloc[0]
            transformed_key = f"transformed_data/fact_taxi_trips/taxi_{formatted_date}.csv"
        case "weather":
            formatted_date = dataframe["datetime"].dt.strftime("%Y-%m-%d").iloc[0]
            transformed_key = f"transformed_data/dim_weather/weather_{formatted_date}.csv"
        case _:
            raise ValueError("file_type must be either 'taxi' or 'weather'")

    # 1. Upload transformed data
    _upload_dataframe_to_s3(s3, bucket, dataframe, transformed_key)

    # 2. Move raw file to archive
    source_key = f"raw_data/to_processed/{file_type}/{file_type}_{formatted_date}.json"
    target_key = f"raw_data/processed/{file_type}/{file_type}_{formatted_date}.json"

    _move_file_on_s3(s3, bucket, source_key, target_key)


In [ ]:
# lambda_function.py

import boto3
import json

import pandas as pd

from configs import (
    BUCKET,
    DIM_COMPANY_FILE_PATH,
    DIM_PAYMENT_TYPE_FILE_PATH,
    RAW_TAXI_FOLDER,
    RAW_WEATHER_FOLDER
)
from functions import(
    update_dim_table,
    update_fact_taxi_trips_with_dim_data,
    read_file_from_s3,
    transform_taxi,
    transform_weather,
    upload_and_archive_on_s3,
    upload_dim_to_s3
)


def process_taxi_data(s3, dim_payment_type: pd.DataFrame, dim_company: pd.DataFrame):
    """Process and transform daily Taxi data.

    1. Download the raw Taxi data.
    2. Read and transform it.
    3. Update the Payment Type and Company tables.
    4. Update Taxi data with Payment Type and Company ids.
    5. Upload the Payment Type and Company to S3.
    6. Upload and archive Taxi data.

    :param s3:                  S3 client.
    :param dim_payment_type:    Payment Type Dimension table.
    :param dim_company:         Company Dimension table.
    """
    for file in s3.list_objects(Bucket=BUCKET, Prefix=RAW_TAXI_FOLDER)["Contents"]:
        taxi_key = file["Key"]
        taxi_raw_file_name = file["Key"].split("/")[-1]
        
        if taxi_raw_file_name.split(".")[-1] == "json":        
            taxi_raw_content = read_file_from_s3(s3, BUCKET, taxi_key, "json")
            fact_taxi_trips = transform_taxi(taxi_raw_content)

            dim_payment_type_updated = update_dim_table(fact_taxi_trips, dim_payment_type, "payment_type")            
            dim_company_updated =  update_dim_table(fact_taxi_trips, dim_company, "company")

            fact_taxi_trips_updated = update_fact_taxi_trips_with_dim_data(
                fact_taxi_trips,
                dim_payment_type_updated,
                dim_company_updated
            )

            upload_dim_to_s3(s3, BUCKET, "payment_type", dim_payment_type_updated)
            upload_dim_to_s3(s3, BUCKET, "company", dim_company_updated)

            upload_and_archive_on_s3(s3, fact_taxi_trips_updated, BUCKET, "taxi")


def process_weather_data(s3):
    """Process and transform daily Weather data.

    1. Download the raw Weather data.
    2. Read and transform it.
    3. Upload and archive the files.
    """
    for file in s3.list_objects(Bucket=BUCKET, Prefix=RAW_WEATHER_FOLDER)["Contents"]:
        weather_key = file["Key"]
        weather_raw_file_name = file["Key"].split("/")[-1]
        
        if weather_raw_file_name.split(".")[-1] == "json":
            weather_raw_content = read_file_from_s3(s3, BUCKET, weather_key, "json")
            dim_weather = transform_weather(weather_raw_content)

            upload_and_archive_on_s3(s3, dim_weather, BUCKET, "weather")


def lambda_handler(event, context):
    s3 = boto3.client("s3")

    dim_payment_type = read_file_from_s3(s3, BUCKET, DIM_PAYMENT_TYPE_FILE_PATH, "csv")
    dim_company = read_file_from_s3(s3, BUCKET, DIM_COMPANY_FILE_PATH, "csv")

    process_taxi_data(s3, dim_payment_type, dim_company)
    process_weather_data(s3)

    print("All files have been processed.")
